# 16 — Ensambles

¿Combinar el modelo final (CatBoost P6b, 0,9365) con otros modelos mejora la accuracy?

**Material:**
- `material/Arboles/Árboles.pdf`: slide 56 ("Voting / Stacking: técnicas de promedio o apilamiento de diferentes clasificadores"), slides 73–74 (Voting / Stacking) y **slide 75 (Hill Climber)**.
- `material/modelos de ensamble/S8_Ensambles.ipynb`: `VotingClassifier`, voto *soft* = promedio de probabilidades; aclara que Stacking "no forma parte de la versión actual de la clase".

**Plan:**
1. Calcular **una vez** las probabilidades *out-of-fold* (OOF) de 6 modelos con la CV de siempre, y guardarlas en disco.
2. Correlación entre modelos y % de desacuerdo.
3. Ensambles sobre esas probabilidades. Todo lo que se elija mirando datos se elige con 4 folds y se evalúa en el quinto, rotando.

Comparación pareada contra CatBoost solo, con el criterio de siempre.

In [1]:
import sys
sys.path.append("..")

import os
import time
import warnings
import numpy as np
import pandas as pd
from joblib import Memory

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer, StandardScaler, TargetEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from catboost import CatBoostClassifier

from features import (
    aplanar_anidadas, agregar_logs, agregar_features_contenido, agregar_categoricas_crudas,
    CapeoPorMoneda, VendedorEncoder, PrediccionOOF, armar_modelo_texto,
    COLUMNAS_CONTENIDO, COLUMNAS_NUMERICAS, COLUMNAS_CAT_CRUDAS, INDICES_CAT,
)

SEED = 42

train = pd.read_json("../data/train_data.jsonlines", lines=True)
y = (train["condition"] == "new").astype(int)
train = agregar_categoricas_crudas(agregar_features_contenido(aplanar_anidadas(train)))

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
folds = list(cv.split(train, y))
fold_de_cada_fila = np.empty(len(train), dtype=int)
for k, (_, indices_validacion) in enumerate(folds):
    fold_de_cada_fila[indices_validacion] = k + 1

carpeta_lab = ".cache_laboratorio"
memoria_texto = Memory(location=carpeta_lab, verbose=0)
archivo_oof = os.path.join(carpeta_lab, "oof_ensambles.csv")

## 1. Los 6 modelos

El modelo de texto (`PrediccionOOF`) es el mismo en los 6, sobre los mismos folds. Va envuelto igual que en `laboratorio.ipynb` (un `Pipeline` con `memory`), así que reusa la caché y no se recalcula.

| Modelo | Configuración | De dónde sale |
|---|---|---|
| CatBoost P6b | igual que `armar_pipeline_final()` | `features.py` / laboratorio (0,9365) |
| HistGB (Base) | Base del laboratorio | laboratorio (0,9339) |
| Regresión logística | `C=1`, `penalty="l1"`, `solver="liblinear"` | corrida parcial de `13_comparacion_modelos` |
| LDA | `solver="lsqr"`, sin *shrinkage* | corrida parcial de la 13 |
| kNN | `n_neighbors=15`, `weights="distance"` | corrida parcial de la 13 |
| Random Forest | 200 árboles, `max_depth=20`, `max_features="sqrt"` | `train.ipynb` (la 13 se cortó antes de llegar al RF) |

Los 4 últimos usan el preprocesador de la 13 (escalado, imputación de fotos por mediana + indicador, `VendedorEncoder` + `TargetEncoder` + one-hot), porque no aceptan categóricas crudas.

In [2]:
def texto_cacheado():
    return Pipeline([
        ("oof", PrediccionOOF(armar_modelo_texto(), random_state=SEED)),
        ("identidad", "passthrough"),
    ], memory=memoria_texto)


def con_capeo_y_logs(preprocesador, modelo):
    return Pipeline([
        ("capeo", CapeoPorMoneda(columnas=["price", "base_price"], columna_grupo="currency_id")),
        ("logs", FunctionTransformer(agregar_logs)),
        ("preprocesador", preprocesador),
        ("modelo", modelo),
    ])


# CatBoost P6b: mismo ColumnTransformer que armar_pipeline_final(), con el texto cacheado
preprocesador_catboost = ColumnTransformer([
    ("num", "passthrough", COLUMNAS_NUMERICAS),
    ("tfidf", TfidfVectorizer(max_features=500), "title"),
    ("texto", texto_cacheado(), ["title", "warranty_texto"]),
    ("cat_crudas", "passthrough", COLUMNAS_CAT_CRUDAS),
], sparse_threshold=0)

# HistGB: la Base del laboratorio
columnas_categoricas = ["listing_type_id", "buying_mode", "shipping_mode", "seller_state"]
preprocesador_histgb = ColumnTransformer([
    ("onehot", OneHotEncoder(handle_unknown="ignore"), columnas_categoricas),
    ("num", "passthrough", COLUMNAS_NUMERICAS),
    ("categoria", TargetEncoder(target_type="binary", random_state=SEED), ["category_id"]),
    ("tfidf", TfidfVectorizer(max_features=500), "title"),
    ("texto", texto_cacheado(), ["title", "warranty_texto"]),
    ("vendedor", VendedorEncoder(incluir_target=True), ["seller_id", "log_price"]),
], sparse_threshold=0)

# Regresión logística, LDA, kNN, Random Forest: el preprocesador de 13_comparacion_modelos
columnas_escala = ["log_price", "log_available_quantity", "sold_quantity", "initial_quantity",
                   "n_fotos", "n_atributos", "n_variaciones", "n_metodos_pago"]
columnas_bool = ["automatic_relist", "tiene_video", "tiene_garantia", "free_shipping", "local_pick_up",
                 "accepts_mercadopago", "es_tienda_oficial", "tiene_descuento", "tiene_oferta", "tiene_parent_item"]


def preprocesador_comparacion():
    return ColumnTransformer([
        ("onehot", OneHotEncoder(handle_unknown="ignore", drop="first"), columnas_categoricas),
        ("escala", StandardScaler(), columnas_escala),
        ("bool", "passthrough", columnas_bool),
        ("contenido", Pipeline([
            ("imputacion", SimpleImputer(strategy="median", add_indicator=True)),
            ("escala", StandardScaler()),
        ]), COLUMNAS_CONTENIDO),
        ("tfidf", TfidfVectorizer(max_features=500), "title"),
        ("texto", texto_cacheado(), ["title", "warranty_texto"]),
        ("vendedor", VendedorEncoder(incluir_target=True), ["seller_id", "log_price"]),
        ("categoria", TargetEncoder(target_type="binary", random_state=SEED), ["category_id"]),
    ], sparse_threshold=0)


modelos = {
    "catboost": (con_capeo_y_logs(preprocesador_catboost, CatBoostClassifier(random_state=SEED, verbose=0)),
                 {"modelo__cat_features": INDICES_CAT}),
    "histgb": (con_capeo_y_logs(preprocesador_histgb,
                                HistGradientBoostingClassifier(learning_rate=0.1, max_depth=None, max_iter=200,
                                                               random_state=SEED)), None),
    "logistica": (con_capeo_y_logs(preprocesador_comparacion(),
                                   LogisticRegression(C=1, penalty="l1", solver="liblinear", max_iter=1000,
                                                      random_state=SEED)), None),
    "lda": (con_capeo_y_logs(preprocesador_comparacion(), LinearDiscriminantAnalysis(solver="lsqr")), None),
    "knn": (con_capeo_y_logs(preprocesador_comparacion(), KNeighborsClassifier(n_neighbors=15, weights="distance")), None),
    "random_forest": (con_capeo_y_logs(preprocesador_comparacion(),
                                       RandomForestClassifier(n_estimators=200, max_depth=20, max_features="sqrt",
                                                              random_state=SEED)), None),
}

## Probabilidades *out-of-fold*

`cross_val_predict(method="predict_proba")` (material: `S8_Ensambles.ipynb`): cada fila recibe P(`new`) de un modelo que **no la vio**, entrenado con los otros 4 folds. Se guardan en `.cache_laboratorio/oof_ensambles.csv`, con una columna `fold` que dice en qué fold de validación cayó cada fila. Si el archivo ya existe, se lee y no se recalcula.

Chequeo: la accuracy de cada fold calculada desde estas probabilidades tiene que coincidir con la del registro del laboratorio, para CatBoost y para el HistGB.

In [3]:
if os.path.exists(archivo_oof):
    oof = pd.read_csv(archivo_oof)
    print("Leído de", archivo_oof)
else:
    oof = pd.DataFrame({"fold": fold_de_cada_fila, "y": y.to_numpy()})
    for nombre, (pipeline, parametros_fit) in modelos.items():
        inicio = time.time()
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            oof[nombre] = cross_val_predict(pipeline, train, y, cv=cv, method="predict_proba", n_jobs=5,
                                            params=parametros_fit)[:, 1]
        print(f"{nombre}: {(time.time() - inicio) / 60:.1f} min")
    oof.to_csv(archivo_oof, index=False)
    print("Guardado en", archivo_oof)

nombres_modelos = [c for c in oof.columns if c not in ("fold", "y")]

Leído de .cache_laboratorio/oof_ensambles.csv


In [4]:
def accuracy_por_fold(probabilidad, umbral=0.5):
    acierto = (probabilidad >= umbral).astype(int) == oof["y"]
    return acierto.groupby(oof["fold"]).mean().to_numpy()


tabla_individual = pd.DataFrame({nombre: accuracy_por_fold(oof[nombre]) for nombre in nombres_modelos},
                                index=[f"fold_{k}" for k in range(1, 6)]).T
tabla_individual["media"] = tabla_individual.mean(axis=1)
tabla_individual["desvio"] = tabla_individual.iloc[:, :5].std(axis=1, ddof=0)
tabla_individual = tabla_individual.sort_values("media", ascending=False)

registro = pd.read_csv(os.path.join(carpeta_lab, "resultados.csv")).set_index("nombre")
columnas_folds = ["fold_1", "fold_2", "fold_3", "fold_4", "fold_5"]
print("CatBoost == registro 'Final':",
      np.allclose(tabla_individual.loc["catboost", columnas_folds].to_numpy(dtype=float),
                  registro.loc["Final (armar_pipeline_final de features.py)", columnas_folds].to_numpy(dtype=float)))
print("HistGB == registro 'Base':",
      np.allclose(tabla_individual.loc["histgb", columnas_folds].to_numpy(dtype=float),
                  registro.loc["Base", columnas_folds].to_numpy(dtype=float)))
tabla_individual.round(4)

CatBoost == registro 'Final': True
HistGB == registro 'Base': True


,fold_1,fold_2,fold_3,fold_4,fold_5,media,desvio
catboost,0.9372,0.9332,0.9366,0.9381,0.9375,0.9365,0.0017
histgb,0.9332,0.9308,0.9359,0.9349,0.9349,0.9339,0.0018
random_forest,0.9259,0.9236,0.9269,0.9276,0.9281,0.9264,0.0016
lda,0.9239,0.9190,0.9231,0.9239,0.9236,0.9227,0.0019
logistica,0.9228,0.9189,0.9234,0.9232,0.9221,0.9221,0.0017
knn,0.8957,0.8897,0.8995,0.8988,0.8977,0.8963,0.0035


## 2. ¿Qué tan distintos son los modelos?

Un ensamble solo puede mejorar si los modelos **se equivocan en filas distintas** (`S8_Ensambles.ipynb`: "un ensamble necesita modelos suficientemente buenos y, al mismo tiempo, suficientemente diversos").
- **Correlación** entre las probabilidades: cerca de 1 = dicen casi lo mismo.
- **% de desacuerdo**: en qué % de filas predicen clases distintas (con umbral 0,5). Es el máximo de filas que un ensamble de esos dos podría "dar vuelta".

In [5]:
oof[nombres_modelos].corr().round(3)

,catboost,histgb,logistica,lda,knn,random_forest
catboost,1.000,0.991,0.977,0.973,0.917,0.974
histgb,0.991,1.000,0.977,0.972,0.916,0.975
logistica,0.977,0.977,1.000,0.992,0.917,0.970
lda,0.973,0.972,0.992,1.000,0.905,0.962
knn,0.917,0.916,0.917,0.905,1.000,0.944
random_forest,0.974,0.975,0.970,0.962,0.944,1.000


In [6]:
clases = {nombre: (oof[nombre] >= 0.5).astype(int) for nombre in nombres_modelos}
desacuerdo = pd.DataFrame(
    [[(clases[a] != clases[b]).mean() for b in nombres_modelos] for a in nombres_modelos],
    index=nombres_modelos, columns=nombres_modelos,
)
(desacuerdo * 100).round(1)

,catboost,histgb,logistica,lda,knn,random_forest
catboost,0.0,2.0,3.7,3.7,7.6,2.8
histgb,2.0,0.0,3.7,3.7,7.7,2.8
logistica,3.7,3.7,0.0,1.7,8.0,4.2
lda,3.7,3.7,1.7,0.0,8.1,4.1
knn,7.6,7.7,8.0,8.1,0.0,6.9
random_forest,2.8,2.8,4.2,4.1,6.9,0.0


In [7]:
# Cuando CatBoost se equivoca, ¿qué % de esos errores acierta cada otro modelo?
error_catboost = clases["catboost"] != oof["y"]
print("Errores de CatBoost:", error_catboost.sum())
for nombre in nombres_modelos:
    if nombre != "catboost":
        acierta = (clases[nombre] == oof["y"])[error_catboost].mean()
        print(f"  {nombre:14s} acierta el {acierta:.1%} de los errores de CatBoost")

Errores de CatBoost: 4442
  histgb         acierta el 14.0% de los errores de CatBoost
  logistica      acierta el 18.0% de los errores de CatBoost
  lda            acierta el 18.6% de los errores de CatBoost
  knn            acierta el 28.5% de los errores de CatBoost
  random_forest  acierta el 14.4% de los errores de CatBoost


**Lectura.** El HistGB es casi un clon de CatBoost (correlación 0,991, solo 2% de desacuerdo). Los modelos más "distintos" (kNN, que acierta el 29% de los errores de CatBoost) son también los más flojos. Es el dilema de S8: diversidad sí, pero con modelos suficientemente buenos. Expectativa: ganancias chicas en el mejor caso.

## 3. Ensambles

**Protocolo honesto.** Para cada fold k: lo que haya que elegir (pesos, meta-modelo) se elige con las probabilidades OOF de **los otros 4 folds** y se aplica al fold k. Quedan 5 accuracies honestas, que se comparan **fold por fold contra CatBoost solo** con el criterio de siempre (mejora real: gana 5/5, o 4/5 con media > desvío de la diferencia).

`evaluar_rotando(nombre, combinar)` hace la rotación: `combinar(entrenamiento, prueba)` recibe las filas OOF de los 4 folds y del fold k, y devuelve la probabilidad combinada del fold k y lo que eligió.

In [8]:
from sklearn.linear_model import LogisticRegression

accuracy_catboost = tabla_individual.loc["catboost", columnas_folds].to_numpy(dtype=float)
resultados_ensambles = {}


def evaluar_rotando(nombre, combinar):
    accuracies, elegidos = [], []
    for k in range(1, 6):
        entrenamiento = oof[oof["fold"] != k]
        prueba = oof[oof["fold"] == k]
        probabilidad, elegido = combinar(entrenamiento, prueba)
        accuracies.append(((probabilidad >= 0.5).astype(int) == prueba["y"].to_numpy()).mean())
        elegidos.append(elegido)
    accuracies = np.array(accuracies)
    diferencias = accuracies - accuracy_catboost
    gana = (diferencias > 0).sum()
    resultados_ensambles[nombre] = {
        "accuracy_media": accuracies.mean(),
        "accuracy_std": accuracies.std(),
        "folds_gana": gana,
        "media_diferencia": diferencias.mean(),
        "desvio_diferencia": diferencias.std(),
        "mejora_real": gana == 5 or (gana == 4 and diferencias.mean() > diferencias.std()),
        "elegido_por_fold": elegidos,
    }
    print(f"{nombre}: accuracy {accuracies.mean():.4f} ± {accuracies.std():.4f} | dif. por fold {np.round(diferencias, 4)}"
          f" | gana {gana}/5 | -> {'MEJORA REAL' if resultados_ensambles[nombre]['mejora_real'] else 'no pasa'}")

### (a) Voting soft HistGB + CatBoost

Promedio simple de las dos probabilidades (fórmula del voto *soft* de `S8_Ensambles.ipynb`). No se elige nada.

In [9]:
evaluar_rotando("(a) promedio CatBoost + HistGB",
                lambda entrenamiento, prueba: (((prueba["catboost"] + prueba["histgb"]) / 2).to_numpy(), "50/50"))

(a) promedio CatBoost + HistGB: accuracy 0.9359 ± 0.0017 | dif. por fold [-0.0006 -0.0006 -0.0001 -0.0012 -0.0004] | gana 0/5 | -> no pasa


### (b) Voting ponderado: CatBoost con cada uno de los otros

Probabilidad = w · CatBoost + (1 − w) · otro. El w se elige en una grilla de 0 a 1 (de a 0,05) con los 4 folds de entrenamiento. Ante empate se elige el w más alto (lo más parecido a CatBoost solo). El parámetro `weights` de `VotingClassifier` aparece en la documentación que muestra S8, pero no se trabaja en clase.

In [10]:
grilla_w = np.round(np.arange(1.0, -0.001, -0.05), 2)


def ponderado_con(otro):
    def combinar(entrenamiento, prueba):
        accuracy_por_w = [
            ((w * entrenamiento["catboost"] + (1 - w) * entrenamiento[otro] >= 0.5).astype(int) == entrenamiento["y"]).mean()
            for w in grilla_w
        ]
        w = grilla_w[int(np.argmax(accuracy_por_w))]
        return (w * prueba["catboost"] + (1 - w) * prueba[otro]).to_numpy(), w
    return combinar


for otro in ["histgb", "random_forest", "logistica", "lda", "knn"]:
    evaluar_rotando(f"(b) CatBoost ponderado con {otro}", ponderado_con(otro))
    print("    w de CatBoost elegido en cada rotación:", resultados_ensambles[f"(b) CatBoost ponderado con {otro}"]["elegido_por_fold"])

(b) CatBoost ponderado con histgb: accuracy 0.9366 ± 0.0017 | dif. por fold [ 0.0001  0.0001  0.0003  0.0001 -0.0004] | gana 4/5 | -> no pasa
    w de CatBoost elegido en cada rotación: [np.float64(0.9), np.float64(0.95), np.float64(0.9), np.float64(0.8), np.float64(0.8)]
(b) CatBoost ponderado con random_forest: accuracy 0.9364 ± 0.0017 | dif. por fold [-0.0001  0.     -0.0003 -0.0004  0.0001] | gana 1/5 | -> no pasa
    w de CatBoost elegido en cada rotación: [np.float64(0.85), np.float64(1.0), np.float64(0.9), np.float64(0.95), np.float64(0.95)]
(b) CatBoost ponderado con logistica: accuracy 0.9363 ± 0.0017 | dif. por fold [-0.0004  0.      0.      0.     -0.0006] | gana 0/5 | -> no pasa
    w de CatBoost elegido en cada rotación: [np.float64(0.9), np.float64(1.0), np.float64(1.0), np.float64(1.0), np.float64(0.9)]
(b) CatBoost ponderado con lda: accuracy 0.9364 ± 0.0017 | dif. por fold [-0.0004  0.     -0.0001  0.     -0.0003] | gana 0/5 | -> no pasa
    w de CatBoost elegido en ca

(b) CatBoost ponderado con knn: accuracy 0.9364 ± 0.0016 | dif. por fold [ 0.0001  0.0002  0.0001  0.0001 -0.0014] | gana 4/5 | -> no pasa
    w de CatBoost elegido en cada rotación: [np.float64(0.9), np.float64(0.85), np.float64(0.9), np.float64(0.9), np.float64(0.8)]


### (c) Hill Climber (slide 75 de `Árboles.pdf`)

El pseudocódigo de la slide: pesos iguales; hasta 10.000 iteraciones; para cada modelo, probar su peso ± 0,01; normalizar; quedarse con el cambio si la métrica (`compute_cv`, acá accuracy sobre los 4 folds) mejora.

**Problema del pseudocódigo:** normaliza con `softmax`. Aplicado a pesos que ya suman 1, `softmax` los aplasta hacia "todos iguales": empujando un peso hacia arriba 100.000 veces, con 6 modelos llega como máximo a 0,168 (desde 0,167). Por eso se miden dos versiones:
- **(c1) literal**, con `softmax` (en la práctica, casi el promedio simple de los 6);
- **(c2) normalización simple**: pesos ≥ 0 divididos por su suma (la versión habitual del método).

Si en una vuelta completa ningún cambio mejora, se corta: las iteraciones restantes repetirían exactamente lo mismo, así que el resultado es idéntico al de 10.000 iteraciones.

In [11]:
def softmax(w):
    e = np.exp(w - w.max())
    return e / e.sum()


def normalizar_simple(w):
    w = np.clip(w, 0, None)
    return w / w.sum() if w.sum() > 0 else None


def hill_climber(probabilidades, y_real, normalizar, iteraciones=10000):
    n_modelos = probabilidades.shape[1]
    pesos = np.ones(n_modelos) / n_modelos
    mejor = ((probabilidades @ pesos >= 0.5).astype(int) == y_real).mean()
    for _ in range(iteraciones):
        mejoro = False
        for i in range(n_modelos):
            for delta in [+0.01, -0.01]:
                nuevos = pesos.copy()
                nuevos[i] += delta
                nuevos = normalizar(nuevos)
                if nuevos is None:
                    continue
                valor = ((probabilidades @ nuevos >= 0.5).astype(int) == y_real).mean()
                if valor > mejor:
                    mejor, pesos, mejoro = valor, nuevos, True
        if not mejoro:
            break
    return pesos


def hill_climber_combinar(normalizar):
    def combinar(entrenamiento, prueba):
        pesos = hill_climber(entrenamiento[nombres_modelos].to_numpy(), entrenamiento["y"].to_numpy(), normalizar)
        return prueba[nombres_modelos].to_numpy() @ pesos, dict(zip(nombres_modelos, np.round(pesos, 3)))
    return combinar


evaluar_rotando("(c1) Hill Climber literal (softmax)", hill_climber_combinar(softmax))
evaluar_rotando("(c2) Hill Climber normalización simple", hill_climber_combinar(normalizar_simple))
for nombre in ["(c1) Hill Climber literal (softmax)", "(c2) Hill Climber normalización simple"]:
    print(nombre, "- pesos por rotación:")
    for pesos in resultados_ensambles[nombre]["elegido_por_fold"]:
        print("   ", pesos)

(c1) Hill Climber literal (softmax): accuracy 0.9315 ± 0.0017 | dif. por fold [-0.0064 -0.0048 -0.0041 -0.0054 -0.0047] | gana 0/5 | -> no pasa


(c2) Hill Climber normalización simple: accuracy 0.9322 ± 0.0016 | dif. por fold [-0.0061 -0.0035 -0.0028 -0.0046 -0.0046] | gana 0/5 | -> no pasa
(c1) Hill Climber literal (softmax) - pesos por rotación:
    {'catboost': np.float64(0.167), 'histgb': np.float64(0.165), 'logistica': np.float64(0.167), 'lda': np.float64(0.167), 'knn': np.float64(0.167), 'random_forest': np.float64(0.167)}
    {'catboost': np.float64(0.167), 'histgb': np.float64(0.167), 'logistica': np.float64(0.167), 'lda': np.float64(0.167), 'knn': np.float64(0.165), 'random_forest': np.float64(0.167)}
    {'catboost': np.float64(0.166), 'histgb': np.float64(0.166), 'logistica': np.float64(0.168), 'lda': np.float64(0.166), 'knn': np.float64(0.166), 'random_forest': np.float64(0.166)}
    {'catboost': np.float64(0.167), 'histgb': np.float64(0.167), 'logistica': np.float64(0.167), 'lda': np.float64(0.167), 'knn': np.float64(0.165), 'random_forest': np.float64(0.167)}
    {'catboost': np.float64(0.167), 'histgb': np.float6

### (d) Stacking con regresión logística como meta-modelo

Con los 4 folds, una regresión logística aprende a combinar las 6 probabilidades OOF (una columna por modelo) para predecir `condition`, y predice el quinto. Como las entradas son *out-of-fold*, el meta-modelo no ve probabilidades "memorizadas" (misma lógica que `PrediccionOOF` y que el `StackingClassifier` de `train.ipynb`). Material: slide 56 de `Árboles.pdf` ("apilamiento"); S8 aclara que no es parte de la versión actual de la clase.

In [12]:
def stacking(entrenamiento, prueba):
    meta = LogisticRegression(max_iter=1000, random_state=SEED)
    meta.fit(entrenamiento[nombres_modelos], entrenamiento["y"])
    return meta.predict_proba(prueba[nombres_modelos])[:, 1], dict(zip(nombres_modelos, np.round(meta.coef_[0], 2)))


evaluar_rotando("(d) Stacking (meta-modelo: regresión logística)", stacking)
print("Coeficientes del meta-modelo por rotación:")
for coeficientes in resultados_ensambles["(d) Stacking (meta-modelo: regresión logística)"]["elegido_por_fold"]:
    print("   ", coeficientes)

(d) Stacking (meta-modelo: regresión logística): accuracy 0.9364 ± 0.0012 | dif. por fold [-0.0009  0.0009  0.0008 -0.0009 -0.0003] | gana 2/5 | -> no pasa
Coeficientes del meta-modelo por rotación:
    {'catboost': np.float64(4.35), 'histgb': np.float64(1.81), 'logistica': np.float64(0.72), 'lda': np.float64(-1.61), 'knn': np.float64(1.07), 'random_forest': np.float64(2.15)}
    {'catboost': np.float64(4.51), 'histgb': np.float64(1.85), 'logistica': np.float64(0.85), 'lda': np.float64(-1.78), 'knn': np.float64(1.08), 'random_forest': np.float64(1.96)}
    {'catboost': np.float64(4.62), 'histgb': np.float64(1.63), 'logistica': np.float64(0.65), 'lda': np.float64(-1.58), 'knn': np.float64(0.91), 'random_forest': np.float64(2.21)}
    {'catboost': np.float64(4.51), 'histgb': np.float64(1.86), 'logistica': np.float64(0.81), 'lda': np.float64(-1.75), 'knn': np.float64(0.85), 'random_forest': np.float64(2.14)}
    {'catboost': np.float64(4.45), 'histgb': np.float64(1.93), 'logistica': np.fl

## Tabla final

In [13]:
tabla_ensambles = pd.DataFrame(resultados_ensambles).T.drop(columns="elegido_por_fold")
fila_catboost = pd.DataFrame({"accuracy_media": [accuracy_catboost.mean()], "accuracy_std": [accuracy_catboost.std()]},
                             index=["CatBoost solo (referencia)"])
pd.concat([fila_catboost, tabla_ensambles]).sort_values("accuracy_media", ascending=False)

,accuracy_media,accuracy_std,folds_gana,media_diferencia,desvio_diferencia,mejora_real
(b) CatBoost ponderado con histgb,0.936571,0.001668,4,0.000029,0.000242,False
CatBoost solo (referencia),0.936543,0.001733,NaN,NaN,NaN,NaN
(d) Stacking (meta-modelo: regresión logística),0.936443,0.00125,2,-0.0001,0.000788,False
(b) CatBoost ponderado con lda,0.9364,0.001679,0,-0.000143,0.00015,False
(b) CatBoost ponderado con random_forest,0.936386,0.001655,1,-0.000157,0.000183,False
(b) CatBoost ponderado con knn,0.936371,0.00162,4,-0.000171,0.000595,False
(b) CatBoost ponderado con logistica,0.936343,0.001653,0,-0.0002,0.000249,False
(a) promedio CatBoost + HistGB,0.935929,0.001692,0,-0.000614,0.000352,False
(c2) Hill Climber normalización simple,0.932214,0.001582,0,-0.004329,0.001142,False
(c1) Hill Climber literal (softmax),0.931471,0.00168,0,-0.005071,0.000773,False


## Conclusión: **ningún ensamble mejora a CatBoost solo. El modelo final sigue siendo CatBoost P6b.**

| Ensamble | accuracy | vs CatBoost solo (fold por fold) |
|---|---|---|
| **CatBoost solo** | **0,9365 ± 0,0017** | — |
| (b) CatBoost ponderado con HistGB (w ≈ 0,8–0,95) | 0,9366 | 4/5, media +0,00003, desvío 0,0002 → ruido |
| (d) Stacking, meta-modelo regresión logística | 0,9364 | 2/5 |
| (b) CatBoost con LDA / RF / kNN / logística | 0,9363–0,9364 | 0–4/5, media negativa |
| (a) promedio 50/50 CatBoost + HistGB | 0,9359 | **pierde en 5/5** |
| (c2) Hill Climber, normalización simple | 0,9322 | pierde en 5/5 |
| (c1) Hill Climber literal (softmax) | 0,9315 | pierde en 5/5 |

- **(a) y (b):** cuando se deja elegir el peso con 4 folds, el w de CatBoost queda entre 0,8 y 1. O sea, el ensamble "pide" casi solo CatBoost, y lo poco que suma el otro modelo no se sostiene en el fold de prueba. El promedio 50/50 con el HistGB **empeora**: el HistGB es casi un clon de CatBoost pero un poco peor, así que promediar solo diluye.
- **(c) Hill Climber:** la versión literal queda en pesos ≈ 1/6 cada uno, por el `softmax` (lo anticipado). Es casi el promedio de los 6, y con kNN, LDA y logística adentro pierde ~0,5 puntos. La versión con normalización simple tampoco se aleja mucho de pesos iguales: arranca en 1/6 cada uno y, con pasos de 0,01 sobre una métrica "escalonada" como la accuracy, se traba enseguida en un óptimo local. Es una limitación del método tal como está en la slide (arranque uniforme + accuracy como métrica).
- **(d) Stacking:** el meta-modelo le da el mayor coeficiente a CatBoost (≈ 4,5), y combina el resto de forma estable entre rotaciones. Igual no mejora: 2/5, media −0,0001.
- **Por qué no funciona** (lectura, no medida): los otros modelos o son casi iguales a CatBoost (HistGB, correlación 0,991) o son bastante peores (0,90–0,93). Ninguno aporta errores "distintos" **y** suficientemente buenos. Es el caso que describe `S8_Ensambles.ipynb`: "si hay un modelo fuerte/dominante… votar no ayuda porque los otros con los que lo combinamos diluyen su performance".

**Sesgo de selección:** no aplica: ninguna de las 9 pruebas pasó, ni siquiera justo.